In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.utils import *

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

In [6]:
# Load Data from default train and test dataframes
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Define the target column and seperate it from the training data as well as declaring teh x_test values to use
target = "PitNextLap"
x = train.drop(columns=[target])
y = train[target]
x_test = test.copy()

# Defining the categorical features to look at
features = x.select_dtypes(include=["object", "category"]).columns.tolist()

# Defining train test split from sklearn
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Defining the model 
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    early_stopping_rounds=100,
    verbose=200
)

# Fitting the model using the splits
model.fit(
    x_train,
    y_train,
    cat_features=features,
    eval_set=(x_val,y_val),
    use_best_model=True
)

val_predictions = generate_probability_predict(model, x_val)

generate_auc(y_val, val_predictions)

predictions = generate_probability_predict(model, x_test)

generate_submission(predictions, "../submissions/catboost_rawData_baseline_sub.csv")

C:\Users\tyler\AppData\Local\Temp\ipykernel_35980\2924553147.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  features = x.select_dtypes(include=["object", "category"]).columns.tolist()


0:	test: 0.8988855	best: 0.8988855 (0)	total: 181ms	remaining: 3m
200:	test: 0.9359595	best: 0.9359595 (200)	total: 39.8s	remaining: 2m 38s
400:	test: 0.9414575	best: 0.9414575 (400)	total: 1m 17s	remaining: 1m 55s
600:	test: 0.9437912	best: 0.9437912 (600)	total: 1m 54s	remaining: 1m 16s
800:	test: 0.9452160	best: 0.9452160 (800)	total: 2m 31s	remaining: 37.6s
999:	test: 0.9462149	best: 0.9462149 (999)	total: 3m 8s	remaining: 0us

bestTest = 0.9462148678
bestIteration = 999

AUC: 0.9462


,id,PitNextLap
0,439140,0.004969
1,439141,0.004559
2,439142,0.005657
3,439143,0.088592
4,439144,0.815941
...,...,...
188160,627300,0.025577
188161,627301,0.619923
188162,627302,0.759912
188163,627303,0.858433
